# Tokenization — Advanced

> **Section 01 — Topic 01 — advanced**

**Prerequisites:** [foundations.ipynb](./foundations.ipynb)

In the foundations notebook, we explored what tokenization is, why it matters, and how the core algorithms (BPE, WordPiece, Unigram) work at a conceptual level. Now we move into the practical craft of building, training, and evaluating tokenizers for real-world use cases.

Modern language models live and die by their tokenizer. A poorly chosen vocabulary wastes context window on low-information tokens, penalizes non-English speakers with inflated token counts, and struggles with domain-specific text like source code or medical records. This notebook gives you the tools to diagnose these problems and fix them.

## What You'll Learn

- How to **train custom tokenizers** for specific domains using the Hugging Face `tokenizers` library
- **Multilingual tokenization** challenges — the "tokenization tax" and how to measure it
- **Byte-level BPE** — how GPT-2, GPT-4, and Llama handle arbitrary byte sequences with zero unknown tokens
- **Special tokens and chat templates** — how modern chat models structure their input
- **Evaluation metrics** for comparing tokenizer quality: fertility, compression ratio, and coverage

## What You'll Build

- A **domain-specific tokenizer** trained on a custom Python code corpus
- A **multilingual tokenizer** with balanced vocabulary allocation
- A **tokenizer evaluation harness** you can reuse on any tokenizer

## Setup

We need several libraries for this notebook. The `tokenizers` library is Hugging Face's Rust-backed tokenizer training library — it is orders of magnitude faster than pure Python implementations. `sentencepiece` is Google's tokenizer library used by T5, Llama, and many other models. `datasets` gives us convenient access to text corpora for training.

In [ ]:
# Install dependencies
# %pip install tokenizers transformers sentencepiece datasets matplotlib pandas

In [ ]:
import json
import collections
import unicodedata
import tempfile
import os
from typing import Iterator

import matplotlib.pyplot as plt
import pandas as pd

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, processors, decoders
from transformers import AutoTokenizer, PreTrainedTokenizerFast

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

---

## 1. Training Custom Tokenizers with the HF `tokenizers` Library

The Hugging Face `tokenizers` library exposes a modular pipeline for building tokenizers from scratch. Every tokenizer passes through four stages, and understanding this pipeline is essential before you start training:

1. **Normalizer** — Transforms the raw input string before any splitting. Common normalizations include Unicode normalization (NFC, NFD, NFKC), lowercasing, and accent stripping. The normalizer determines what your tokenizer considers "the same character." For example, NFD normalization decomposes `é` into `e` + combining accent, while NFC keeps it as a single codepoint. Your choice here affects vocabulary size and cross-language consistency.

2. **Pre-tokenizer** — Splits the normalized text into "words" or chunks that the model will never merge across. The most common pre-tokenizers split on whitespace, but byte-level pre-tokenizers map the entire input into a byte representation first. The pre-tokenizer defines the boundaries of your merge operations — BPE merges will never cross a pre-token boundary.

3. **Model** — The core algorithm (BPE, WordPiece, or Unigram) that learns and applies subword splits within each pre-token. This is where vocabulary learning happens.

4. **Post-processor** — Adds special tokens (like `[CLS]`, `[SEP]`) and manages the final token IDs. This stage handles the structural formatting that models expect.

In [ ]:
# Prepare a training corpus
# In practice, you would load a real dataset:
#   from datasets import load_dataset
#   ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

sample_corpus = [
    "The transformer architecture revolutionized natural language processing by introducing self-attention mechanisms.",
    "Tokenization is the process of converting raw text into a sequence of tokens that a model can process.",
    "Byte-pair encoding iteratively merges the most frequent pair of adjacent symbols in the training corpus.",
    "Language models predict the probability of the next token given the preceding context window.",
    "Attention mechanisms allow the model to focus on relevant parts of the input sequence.",
    "The vocabulary size is a critical hyperparameter that balances compression efficiency and coverage.",
    "Subword tokenization strikes a balance between character-level and word-level representations.",
    "Pre-training on large corpora gives language models broad knowledge before task-specific fine-tuning.",
    "The embedding layer maps discrete token IDs into continuous vector representations.",
    "Positional encodings provide the model with information about token ordering in the sequence.",
    "Multi-head attention projects queries, keys, and values into multiple subspaces for richer representations.",
    "Layer normalization stabilizes training by normalizing activations across the feature dimension.",
    "Residual connections enable gradient flow through deep transformer networks.",
    "The feed-forward network in each transformer block applies two linear transformations with a nonlinearity.",
    "Masked language modeling trains bidirectional representations by predicting randomly masked tokens.",
    "Causal language modeling trains autoregressive models by predicting each token from its left context.",
    "Beam search explores multiple candidate sequences to find high-probability outputs during generation.",
    "Temperature scaling controls the randomness of token sampling during text generation.",
    "Top-k sampling restricts the candidate pool to the k most probable next tokens.",
    "Nucleus sampling dynamically adjusts the candidate pool based on cumulative probability mass.",
]

# Replicate to create a larger training set
training_corpus = sample_corpus * 50  # 1000 samples

def corpus_iterator() -> Iterator[str]:
    """Yields training examples one at a time."""
    for text in training_corpus:
        yield text

print(f"Training corpus size: {len(training_corpus)} samples")
print(f"Total characters: {sum(len(t) for t in training_corpus):,}")

### 1.1 Training BPE, WordPiece, and Unigram Tokenizers

BPE (Byte-Pair Encoding) is the most widely used subword algorithm in modern LLMs. It starts with individual characters (or bytes) and iteratively merges the most frequent adjacent pair until the desired vocabulary size is reached. The key hyperparameter is `vocab_size` — larger vocabularies mean fewer tokens per sentence (better compression) but more parameters in the embedding matrix. Typical values range from 32,000 (Llama) to 100,000+ (GPT-4).

WordPiece, used by BERT, selects merges that maximize the likelihood of the training corpus rather than just frequency. It uses the `##` prefix for continuation tokens. Unigram takes a top-down approach — starting with a large vocabulary and pruning tokens that contribute least to corpus likelihood. Each algorithm produces different segmentations with different tradeoffs.

In [ ]:
# ---- Train a BPE tokenizer ----
bpe_tokenizer = Tokenizer(models.BPE())
bpe_tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents(),
])
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    min_frequency=2, show_progress=True,
)
bpe_tokenizer.train_from_iterator(corpus_iterator(), trainer=bpe_trainer)
print(f"BPE vocabulary size: {bpe_tokenizer.get_vocab_size()}")

# ---- Train a WordPiece tokenizer ----
wp_tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
wp_tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents(),
])
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

wp_trainer = trainers.WordPieceTrainer(
    vocab_size=8000,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    min_frequency=2,
)
wp_tokenizer.train_from_iterator(corpus_iterator(), trainer=wp_trainer)
print(f"WordPiece vocabulary size: {wp_tokenizer.get_vocab_size()}")

# ---- Train a Unigram tokenizer ----
uni_tokenizer = Tokenizer(models.Unigram())
uni_tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents(),
])
uni_tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()

uni_trainer = trainers.UnigramTrainer(
    vocab_size=8000,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    unk_token="[UNK]",
)
uni_tokenizer.train_from_iterator(corpus_iterator(), trainer=uni_trainer)
print(f"Unigram vocabulary size: {uni_tokenizer.get_vocab_size()}")

In [ ]:
# Compare all three on the same test input
test_text = "Tokenization transforms raw text into subword units for language model processing."

for name, tok in [("BPE", bpe_tokenizer), ("WordPiece", wp_tokenizer), ("Unigram", uni_tokenizer)]:
    enc = tok.encode(test_text)
    print(f"\n{name}:")
    print(f"  Tokens ({len(enc.tokens)}): {enc.tokens}")
    print(f"  IDs: {enc.ids[:15]}...")

In [ ]:
# Quantitative comparison across multiple sentences
test_sentences = [
    "The transformer architecture uses multi-head self-attention mechanisms.",
    "Subword tokenization handles unseen words by decomposing them into known pieces.",
    "Pre-training on diverse corpora improves downstream task performance.",
    "Byte-pair encoding is the most popular subword algorithm in modern LLMs.",
    "Reinforcement learning from human feedback aligns model outputs with preferences.",
]

results = []
for sent in test_sentences:
    results.append({
        "sentence": sent[:55] + "...",
        "BPE": len(bpe_tokenizer.encode(sent).tokens),
        "WordPiece": len(wp_tokenizer.encode(sent).tokens),
        "Unigram": len(uni_tokenizer.encode(sent).tokens),
        "words": len(sent.split()),
    })

df_compare = pd.DataFrame(results)
print(df_compare.to_string(index=False))
print(f"\nAverage tokens — BPE: {df_compare['BPE'].mean():.1f}, "
      f"WordPiece: {df_compare['WordPiece'].mean():.1f}, "
      f"Unigram: {df_compare['Unigram'].mean():.1f}")

---

## 2. Domain-Specific Tokenization

General-purpose tokenizers are trained on broad web text and optimized for average-case performance across many domains. But when you apply them to specialized domains — source code, medical records, legal documents, chemical formulas — they often produce excessively fragmented tokenizations. A single Python keyword like `isinstance` might be split into three or four tokens by a general tokenizer, wasting precious context window.

This fragmentation has real costs. First, it inflates the number of tokens per document, which directly increases compute cost (attention is quadratic in sequence length). Second, it can hurt model performance: if the model needs to "assemble" a keyword from fragments across multiple positions, it has less capacity for reasoning about the code's semantics. Third, it reduces the effective context window — a 4096-token window covers fewer lines of code when each line requires more tokens.

The solution is to train a domain-specific tokenizer on representative data from your target domain. By learning merge rules from domain text, the tokenizer will allocate vocabulary entries to frequently occurring domain patterns.

In [ ]:
# Sample Python code corpus for domain-specific training
code_corpus = [
    "def train_model(self, dataset, epochs=10, learning_rate=0.001):",
    "    self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate)",
    "    for epoch in range(epochs):",
    "        for batch in DataLoader(dataset, batch_size=32, shuffle=True):",
    "            loss = self.compute_loss(batch)",
    "            loss.backward()",
    "            self.optimizer.step()",
    "            self.optimizer.zero_grad()",
    "import torch\nimport torch.nn as nn\nfrom transformers import AutoModel",
    "class TransformerBlock(nn.Module):",
    "    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):",
    "        super().__init__()",
    "        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)",
    "        self.feed_forward = nn.Sequential(",
    "            nn.Linear(d_model, d_ff),",
    "            nn.GELU(),",
    "            nn.Dropout(dropout),",
    "            nn.Linear(d_ff, d_model),",
    "        )",
    "        self.norm1 = nn.LayerNorm(d_model)",
    "        self.norm2 = nn.LayerNorm(d_model)",
    "    def forward(self, x, mask=None):",
    "        attn_output, _ = self.attention(x, x, x, attn_mask=mask)",
    "        x = self.norm1(x + attn_output)",
    "        ff_output = self.feed_forward(x)",
    "        return self.norm2(x + ff_output)",
    "if __name__ == '__main__':",
    "    model = TransformerBlock(d_model=768, n_heads=12)",
    "    input_tensor = torch.randn(32, 128, 768)",
    "    output = model(input_tensor)",
    "    print(f'Output shape: {output.shape}')",
    "def compute_attention_scores(query, key, value, mask=None):",
    "    d_k = query.size(-1)",
    "    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)",
    "    if mask is not None:",
    "        scores = scores.masked_fill(mask == 0, float('-inf'))",
    "    attention_weights = torch.softmax(scores, dim=-1)",
    "    return torch.matmul(attention_weights, value), attention_weights",
]

# Load GPT-2 tokenizer (general-purpose) for comparison
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Tokenize code with general-purpose GPT-2 tokenizer
code_snippet = """def compute_attention_scores(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attention_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attention_weights, value), attention_weights"""

gpt2_tokens = gpt2_tokenizer.tokenize(code_snippet)
print(f"GPT-2 tokenization of code snippet:")
print(f"  Token count: {len(gpt2_tokens)}")
print(f"  Characters per token: {len(code_snippet) / len(gpt2_tokens):.2f}")
print(f"  First 30 tokens: {gpt2_tokens[:30]}...")

In [ ]:
# Train a code-specific BPE tokenizer
code_tokenizer = Tokenizer(models.BPE())
code_tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()])
code_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
code_tokenizer.decoder = decoders.ByteLevel()

code_trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["<|pad|>", "<|unk|>", "<|endoftext|>"],
    min_frequency=2, show_progress=True,
)

# Train on the code corpus (replicated for sufficient statistics)
code_training_data = code_corpus * 100
code_tokenizer.train_from_iterator(iter(code_training_data), trainer=code_trainer)

# Compare tokenization
code_enc = code_tokenizer.encode(code_snippet)
print(f"Code-specific tokenizer:")
print(f"  Vocabulary size: {code_tokenizer.get_vocab_size()}")
print(f"  Token count: {len(code_enc.tokens)}")
print(f"  Characters per token: {len(code_snippet) / len(code_enc.tokens):.2f}")
print(f"  First 30 tokens: {code_enc.tokens[:30]}...")

print(f"\nCompression improvement: {len(gpt2_tokens) / len(code_enc.tokens):.2f}x fewer tokens")

In [ ]:
# Compare compression ratios and fertility across multiple code samples

def compute_fertility(tokenizer_fn, texts):
    """Compute average fertility (tokens per whitespace-delimited word)."""
    total_tokens = sum(len(tokenizer_fn(t)) for t in texts)
    total_words = sum(len(t.split()) for t in texts)
    return total_tokens / total_words if total_words > 0 else 0

test_code_samples = [
    "self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate)",
    "attention_weights = torch.softmax(scores, dim=-1)",
    "class TransformerBlock(nn.Module):",
    "for batch in DataLoader(dataset, batch_size=32, shuffle=True):",
    "output = model(input_tensor)",
]

comparison_results = []
for sample in test_code_samples:
    gpt2_count = len(gpt2_tokenizer.tokenize(sample))
    code_count = len(code_tokenizer.encode(sample).tokens)
    comparison_results.append({
        "code": sample[:50] + ("..." if len(sample) > 50 else ""),
        "GPT-2 tokens": gpt2_count,
        "Code tokenizer": code_count,
        "savings": f"{(1 - code_count / gpt2_count) * 100:.0f}%" if gpt2_count > 0 else "N/A",
    })

df_code = pd.DataFrame(comparison_results)
print(df_code.to_string(index=False))

# Fertility comparison
gpt2_code_fert = compute_fertility(lambda t: gpt2_tokenizer.tokenize(t), test_code_samples)
code_fert = compute_fertility(lambda t: code_tokenizer.encode(t).tokens, test_code_samples)
print(f"\nFertility — GPT-2: {gpt2_code_fert:.3f}, Code-specific: {code_fert:.3f} tokens/word")

---

## 3. Multilingual Tokenization

One of the most significant — and underappreciated — challenges in modern NLP is the **tokenization tax** imposed on non-English languages. When a tokenizer is trained predominantly on English text, its vocabulary is optimized for English morphology and character distributions. Languages with different scripts (Chinese, Arabic, Hindi), agglutinative morphology (Turkish, Finnish, Japanese), or simply different character frequencies get systematically worse tokenization.

The practical impact is severe. The same semantic content — say, "The cat sat on the mat" — might require 7 tokens in English but 15-20 tokens in Hindi or 25+ tokens in Burmese when using an English-centric tokenizer. This means non-English users effectively have a smaller context window, pay more per API call, and experience higher latency. Research by Ahia et al. (2023) and Petrov et al. (2024) has documented these disparities in detail.

Understanding and measuring this tax is the first step toward building more equitable multilingual systems.

In [ ]:
# Compare tokenization across languages using GPT-2 tokenizer
# These sentences all roughly mean "The cat sat on the mat"

multilingual_samples = {
    "English":    "The cat sat on the mat.",
    "Spanish":    "El gato se sentó en la alfombra.",
    "French":     "Le chat était assis sur le tapis.",
    "German":     "Die Katze saß auf der Matte.",
    "Chinese":    "猫坐在垫子上。",
    "Japanese":   "猫はマットの上に座っていた。",
    "Korean":     "고양이가 매트 위에 앉았다.",
    "Arabic":     "جلست القطة على الحصيرة.",
    "Hindi":      "बिल्ली चटाई पर बैठी थी।",
    "Russian":    "Кошка сидела на коврике.",
}

lang_results = []
en_tokens_count = len(gpt2_tokenizer.tokenize(multilingual_samples["English"]))
for lang, text in multilingual_samples.items():
    tokens = gpt2_tokenizer.tokenize(text)
    lang_results.append({
        "Language": lang,
        "Text": text[:35] + ("..." if len(text) > 35 else ""),
        "Characters": len(text),
        "Tokens": len(tokens),
        "Chars/Token": f"{len(text) / len(tokens):.2f}",
        "Tax vs English": f"{len(tokens) / en_tokens_count:.2f}x",
    })

df_lang = pd.DataFrame(lang_results)
print(df_lang.to_string(index=False))

In [ ]:
# Visualize the tokenization tax
fig, ax = plt.subplots(figsize=(12, 6))

languages = [r["Language"] for r in lang_results]
token_counts = [r["Tokens"] for r in lang_results]
colors = ["#2ecc71" if t <= en_tokens_count * 1.2 else "#e74c3c" if t >= en_tokens_count * 2 else "#f39c12"
          for t in token_counts]

bars = ax.barh(languages, token_counts, color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(x=en_tokens_count, color="#2ecc71", linestyle="--", alpha=0.7, label="English baseline")
ax.set_xlabel("Number of Tokens (GPT-2 Tokenizer)")
ax.set_title("Tokenization Tax: Same Meaning, Different Token Counts")
ax.legend()

for bar, count in zip(bars, token_counts):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=10)

plt.tight_layout()
plt.show()

### 3.1 Training a Multilingual Tokenizer with Balanced Sampling

When training a multilingual tokenizer, vocabulary allocation is the central challenge. If you simply concatenate text from all languages and train BPE, the algorithm will naturally allocate more vocabulary entries to whichever language has the most data — typically English. Languages with smaller representation get proportionally fewer dedicated tokens, leading to worse compression.

There are several strategies to address this. **Balanced sampling** ensures each language contributes roughly equally to the training data, regardless of natural corpus sizes — this is what XLM-R does with its exponentially smoothed sampling distribution. **Temperature-based sampling** samples language $l$ with probability $p_l^{1/T}$ where higher temperatures flatten the distribution toward uniform. **Vocabulary budgeting** reserves a fixed fraction of entries for each language or script.

In [ ]:
# Create a balanced multilingual corpus and train a tokenizer
multilingual_corpus = {
    "en": [
        "The transformer architecture has become the dominant paradigm in natural language processing.",
        "Deep learning models require massive amounts of compute and data to train effectively.",
        "Attention mechanisms allow the model to weigh the importance of different input positions.",
        "Transfer learning enables models pre-trained on large corpora to adapt to specific tasks.",
        "The scaling laws predict how model performance improves with more parameters and data.",
    ],
    "es": [
        "La arquitectura transformer se ha convertido en el paradigma dominante del procesamiento del lenguaje natural.",
        "Los modelos de aprendizaje profundo requieren cantidades masivas de computación y datos.",
        "Los mecanismos de atención permiten al modelo ponderar la importancia de diferentes posiciones.",
        "El aprendizaje por transferencia permite adaptar modelos preentrenados a tareas específicas.",
        "Las leyes de escalamiento predicen cómo mejora el rendimiento con más parámetros y datos.",
    ],
    "de": [
        "Die Transformer-Architektur ist zum dominierenden Paradigma in der Verarbeitung natürlicher Sprache geworden.",
        "Deep-Learning-Modelle erfordern massive Mengen an Rechenleistung und Daten zum Training.",
        "Aufmerksamkeitsmechanismen ermöglichen dem Modell die Gewichtung verschiedener Eingabepositionen.",
        "Transferlernen ermöglicht die Anpassung vortrainierter Modelle an spezifische Aufgaben.",
        "Die Skalierungsgesetze sagen vorher wie sich die Leistung mit mehr Parametern verbessert.",
    ],
    "fr": [
        "L'architecture transformer est devenue le paradigme dominant du traitement du langage naturel.",
        "Les modèles d'apprentissage profond nécessitent des quantités massives de calcul et de données.",
        "Les mécanismes d'attention permettent au modèle de pondérer l'importance des différentes positions.",
        "L'apprentissage par transfert permet d'adapter des modèles pré-entraînés à des tâches spécifiques.",
        "Les lois d'échelle prédisent comment les performances s'améliorent avec plus de paramètres.",
    ],
}

# Balance: equal representation per language
balanced_corpus = []
for lang, texts in multilingual_corpus.items():
    balanced_corpus.extend(texts * 200)

# Train multilingual BPE tokenizer
multi_tokenizer = Tokenizer(models.BPE())
multi_tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()])
multi_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
multi_tokenizer.decoder = decoders.ByteLevel()

multi_trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["<|pad|>", "<|unk|>", "<|endoftext|>"],
    min_frequency=2,
)
multi_tokenizer.train_from_iterator(iter(balanced_corpus), trainer=multi_trainer)
print(f"Multilingual tokenizer vocabulary size: {multi_tokenizer.get_vocab_size()}")
print(f"Balanced corpus: {len(balanced_corpus)} samples ({len(balanced_corpus)//4} per language)")

In [ ]:
# Compare our multilingual tokenizer vs GPT-2 and analyze vocabulary allocation
test_multilingual = {
    "en": "The scaling laws predict how model performance improves with more parameters.",
    "es": "Las leyes de escalamiento predicen cómo mejora el rendimiento con más parámetros.",
    "de": "Die Skalierungsgesetze sagen vorher wie sich die Leistung mit mehr Parametern verbessert.",
    "fr": "Les lois d'échelle prédisent comment les performances s'améliorent avec plus de paramètres.",
}

multi_results = []
en_gpt2 = len(gpt2_tokenizer.tokenize(test_multilingual['en']))
en_multi = len(multi_tokenizer.encode(test_multilingual['en']).tokens)

for lang, text in test_multilingual.items():
    gpt2_count = len(gpt2_tokenizer.tokenize(text))
    multi_count = len(multi_tokenizer.encode(text).tokens)
    multi_results.append({
        "Language": lang,
        "GPT-2 tokens": gpt2_count,
        "Multi tokens": multi_count,
        "GPT-2 tax": f"{gpt2_count / en_gpt2:.2f}x",
        "Multi tax": f"{multi_count / en_multi:.2f}x",
    })

print(pd.DataFrame(multi_results).to_string(index=False))

# Vocabulary allocation by Unicode script
def classify_token_script(token: str) -> str:
    scripts = collections.Counter()
    for char in token:
        try:
            script = unicodedata.name(char, "").split()[0]
        except (ValueError, IndexError):
            script = "OTHER"
        scripts[script] += 1
    return scripts.most_common(1)[0][0] if scripts else "EMPTY"

vocab = multi_tokenizer.get_vocab()
script_counts = collections.Counter()
for token in vocab.keys():
    if not token.startswith("<|"):
        script_counts[classify_token_script(token)] += 1

print("\nVocabulary distribution by Unicode script:")
for script, count in script_counts.most_common(10):
    pct = count / sum(script_counts.values()) * 100
    print(f"  {script:20s}: {count:5d} tokens ({pct:5.1f}%)")

---

## 4. Byte-Level BPE

Byte-level BPE is arguably the most important tokenization innovation of the GPT era. Introduced in the GPT-2 paper (Radford et al., 2019), it solves a fundamental problem: how do you tokenize arbitrary text without ever producing an unknown token?

The idea is simple but powerful. Instead of operating on Unicode characters, byte-level BPE operates on raw bytes. Since any text — in any language, with any special characters, emojis, or even binary data — can be represented as a sequence of bytes (values 0-255), the base vocabulary is just 256 entries. BPE merges are then learned on top of this byte representation.

The key insight is that GPT-2 maps each byte to a unique Unicode character for display purposes. Bytes 0x00-0xFF are mapped to printable Unicode characters so the vocabulary appears readable, but under the hood everything is byte sequences. The mapping is invertible, so decoding works perfectly.

The benefit is **zero unknown tokens**. No matter what input you throw at a byte-level BPE tokenizer, it can always fall back to individual bytes. The cost is slightly longer sequences for text with many multi-byte UTF-8 characters (like Chinese, where each character is 3 bytes), but BPE merges usually learn common multi-byte sequences quickly.

In [ ]:
# GPT-2's byte-to-character mapping
def bytes_to_unicode():
    """
    Returns the GPT-2 byte-to-unicode mapping.
    This is the exact function from the GPT-2 source code.
    Maps each byte value (0-255) to a unique unicode character.
    """
    bs = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(c) for c in cs]
    return dict(zip(bs, cs))

byte_map = bytes_to_unicode()
print("GPT-2 byte-to-unicode mapping (first 32 entries):")
for i, (byte_val, char) in enumerate(sorted(byte_map.items())[:32]):
    print(f"  byte {byte_val:3d} (0x{byte_val:02x}) -> '{char}' (U+{ord(char):04X})")

In [ ]:
# Byte-level BPE handles EVERYTHING — no unknown tokens ever
tricky_inputs = [
    "Hello, world!",                          # Standard ASCII
    "café résumé naïve",                       # Accented Latin
    "日本語のテスト",                            # Japanese
    "🤖🧠💡",                                   # Emojis
    "\x00\x01\x02\xff",                       # Raw bytes
    "SELECT * FROM users WHERE id = 42;",      # SQL
    "H₂O + NaOH → NaH₂PO₄",                  # Chemistry
]

print("Byte-level BPE handles everything — no unknown tokens:\n")
for text in tricky_inputs:
    tokens = gpt2_tokenizer.tokenize(text)
    token_ids = gpt2_tokenizer.encode(text)
    has_unk = gpt2_tokenizer.unk_token_id in token_ids if gpt2_tokenizer.unk_token_id is not None else False
    print(f"  Input:  {repr(text)}")
    print(f"  Tokens: {tokens}")
    print(f"  UNK?    {has_unk}\n")

### 4.1 Implementing Byte-Level BPE from Scratch

To truly understand byte-level BPE, let us implement a minimal version from scratch. The algorithm is identical to standard BPE, but the initial vocabulary is the 256 byte values instead of Unicode characters. We will train on a small corpus and watch the merges being learned.

This implementation is deliberately simple and unoptimized — production implementations like the `tokenizers` library use Rust for speed. But seeing the algorithm step by step makes the mechanics clear.

In [ ]:
def train_byte_bpe(texts: list[str], num_merges: int = 50) -> tuple[list[tuple], dict]:
    """
    Train byte-level BPE from scratch.
    
    Returns:
        merges: List of (pair, merged_token) tuples
        vocab: Final vocabulary mapping token_id -> bytes
    """
    # Convert all text to byte sequences
    sequences = [list(text.encode("utf-8")) for text in texts]
    
    # Initial vocabulary: individual bytes
    vocab = {i: bytes([i]) for i in range(256)}
    next_id = 256
    merges = []
    
    for step in range(num_merges):
        # Count all adjacent pairs
        pair_counts = collections.Counter()
        for seq in sequences:
            for i in range(len(seq) - 1):
                pair_counts[(seq[i], seq[i + 1])] += 1
        
        if not pair_counts:
            break
        
        # Find and apply most frequent pair
        (a, b), count = pair_counts.most_common(1)[0]
        new_token = next_id
        vocab[new_token] = vocab[a] + vocab[b]
        merges.append(((a, b), new_token))
        
        # Replace all occurrences in sequences
        new_sequences = []
        for seq in sequences:
            new_seq = []
            i = 0
            while i < len(seq):
                if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
                    new_seq.append(new_token)
                    i += 2
                else:
                    new_seq.append(seq[i])
                    i += 1
            new_sequences.append(new_seq)
        sequences = new_sequences
        next_id += 1
        
        if step < 20:  # Print first 20 merges
            a_repr = vocab[a].decode("utf-8", errors="replace")
            b_repr = vocab[b].decode("utf-8", errors="replace")
            merged_repr = vocab[new_token].decode("utf-8", errors="replace")
            print(f"  Merge {step + 1:3d}: '{a_repr}' + '{b_repr}' -> '{merged_repr}' (count: {count})")
    
    return merges, vocab

print("Training byte-level BPE on sample corpus...\n")
merges, byte_vocab = train_byte_bpe(sample_corpus[:10], num_merges=30)
print(f"\nLearned {len(merges)} merges, final vocab size: {len(byte_vocab)}")

In [ ]:
# Compare byte-level vs character-level base vocabulary sizes

# On our English-only corpus
all_chars = set()
all_bytes = set()
for text in sample_corpus:
    all_chars.update(text)
    all_bytes.update(text.encode("utf-8"))

print("Base vocabulary comparison:")
print(f"  English corpus — chars: {len(all_chars)}, bytes: {len(all_bytes)}")

# On multilingual text — the advantage becomes clear
multilingual_text = " ".join(multilingual_samples.values())
ml_chars = set(multilingual_text)
ml_bytes = set(multilingual_text.encode("utf-8"))

print(f"  Multilingual (10 langs) — chars: {len(ml_chars)}, bytes: {len(ml_bytes)}")
print(f"\nByte-level advantage: character vocab grows {len(ml_chars)/len(all_chars):.1f}x ")
print(f"when adding scripts, but byte vocab is always bounded at 256.")
print(f"Character explosion factor: {len(ml_chars) / len(ml_bytes):.1f}x more base entries needed.")

---

## 5. Tokenizer Configuration and Special Tokens

Modern chat models do not simply tokenize raw text. They use **chat templates** — structured formats that wrap user messages, system prompts, and assistant responses in special token sequences. These special tokens serve as control signals that tell the model "this is a user turn," "this is a system instruction," or "generation should stop here."

Understanding chat templates is essential for anyone working with LLM APIs or fine-tuning models. A mismatched template — using Llama's format with a Mistral model, for instance — can cause severe performance degradation because the model has never seen that token pattern during training.

The key special tokens you will encounter are:
- **BOS (Beginning of Sequence)**: Marks the start of input. Used by Llama, not by GPT.
- **EOS (End of Sequence)**: Marks the end of generation. Every model uses this.
- **PAD**: Used to pad shorter sequences in a batch to equal length.
- **Role markers**: `<|system|>`, `<|user|>`, `<|assistant|>`, `[INST]`, etc.

In [ ]:
# Examine special tokens from popular models
for name in ["gpt2", "bert-base-uncased"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"\n{'=' * 60}")
    print(f"Model: {name}")
    print(f"{'=' * 60}")
    print(f"  Vocab size:  {tok.vocab_size}")
    print(f"  BOS token:   {tok.bos_token!r} (id={tok.bos_token_id})")
    print(f"  EOS token:   {tok.eos_token!r} (id={tok.eos_token_id})")
    print(f"  PAD token:   {tok.pad_token!r} (id={tok.pad_token_id})")
    print(f"  UNK token:   {tok.unk_token!r} (id={tok.unk_token_id})")
    print(f"  SEP token:   {tok.sep_token!r} (id={tok.sep_token_id})")
    print(f"  CLS token:   {tok.cls_token!r} (id={tok.cls_token_id})")
    print(f"  MASK token:  {tok.mask_token!r} (id={tok.mask_token_id})")

In [ ]:
# Chat template formats used by major model families

conversation = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "How do I sort a list in Python?"},
    {"role": "assistant", "content": "You can use the sorted() function or the .sort() method."},
    {"role": "user", "content": "What's the difference between them?"},
]

def format_chatml(messages):
    """ChatML format (OpenAI-style)"""
    output = ""
    for msg in messages:
        output += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    output += "<|im_start|>assistant\n"
    return output

def format_llama3(messages):
    """Llama 3 format"""
    output = "<|begin_of_text|>"
    for msg in messages:
        output += f"<|start_header_id|>{msg['role']}<|end_header_id|>\n\n{msg['content']}<|eot_id|>"
    output += "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return output

def format_mistral(messages):
    """Mistral/Mixtral format"""
    output = "<s>"
    for msg in messages:
        if msg["role"] == "user":
            output += f"[INST] {msg['content']} [/INST]"
        elif msg["role"] == "assistant":
            output += f" {msg['content']}</s>"
        elif msg["role"] == "system":
            output += f"[INST] {msg['content']}\n\n"
    return output

print("=== ChatML Format ===")
print(format_chatml(conversation))
print("\n=== Llama 3 Format ===")
print(format_llama3(conversation))
print("\n=== Mistral Format ===")
print(format_mistral(conversation))

In [ ]:
# Build a custom tokenizer with post-processing for BOS/EOS

chat_tokenizer = Tokenizer(models.BPE())
chat_tokenizer.normalizer = normalizers.NFC()
chat_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
chat_tokenizer.decoder = decoders.ByteLevel()

chat_special_tokens = ["<|pad|>", "<|unk|>", "<|bos|>", "<|eos|>", "<|im_start|>", "<|im_end|>"]

chat_trainer = trainers.BpeTrainer(
    vocab_size=8000, special_tokens=chat_special_tokens, min_frequency=2,
)
chat_tokenizer.train_from_iterator(corpus_iterator(), trainer=chat_trainer)

# Add post-processor for automatic BOS/EOS wrapping
chat_tokenizer.post_processor = processors.TemplateProcessing(
    single="<|bos|> $A <|eos|>",
    pair="<|bos|> $A <|eos|> $B:1 <|eos|>:1",
    special_tokens=[
        ("<|bos|>", chat_tokenizer.token_to_id("<|bos|>")),
        ("<|eos|>", chat_tokenizer.token_to_id("<|eos|>")),
    ],
)

# Test
test_enc = chat_tokenizer.encode("Hello, how are you?")
print(f"Encoded with BOS/EOS:")
print(f"  First 5 tokens: {test_enc.tokens[:5]}")
print(f"  Last 3 tokens:  {test_enc.tokens[-3:]}")
print(f"  BOS is '{test_enc.tokens[0]}', EOS is '{test_enc.tokens[-1]}'")

---

## 6. Tokenizer Evaluation Metrics

How do you know if your tokenizer is good? Vocabulary size alone is not sufficient — you need metrics that capture how efficiently the tokenizer compresses text, how well it handles out-of-vocabulary content, and how equitable it is across domains and languages.

The three most important evaluation metrics are:

1. **Fertility (tokens per word)**: The average number of tokens produced per whitespace-delimited word. Lower is better. A fertility of 1.0 means every word maps to a single token. General-purpose English tokenizers typically achieve 1.2-1.5 on English text.

2. **Compression ratio**: The ratio of characters (or bytes) to tokens. Higher is better — it means each token carries more information. Byte-level BPE tokenizers typically achieve compression ratios of 3-5 characters per token on English.

3. **Unknown token rate**: The fraction of tokens that map to the UNK token. For byte-level BPE this is always 0%, but for character-level or word-level tokenizers it can be significant on out-of-domain text.

In [ ]:
class TokenizerEvaluator:
    """Comprehensive tokenizer evaluation harness."""
    
    def __init__(self, name: str, encode_fn, unk_token=None):
        self.name = name
        self.encode_fn = encode_fn  # text -> list of token strings
        self.unk_token = unk_token
    
    def fertility(self, texts: list[str]) -> float:
        """Average tokens per whitespace-delimited word."""
        total_tokens = sum(len(self.encode_fn(t)) for t in texts)
        total_words = sum(len(t.split()) for t in texts)
        return total_tokens / total_words if total_words > 0 else 0
    
    def compression_ratio(self, texts: list[str]) -> float:
        """Average characters per token."""
        total_chars = sum(len(t) for t in texts)
        total_tokens = sum(len(self.encode_fn(t)) for t in texts)
        return total_chars / total_tokens if total_tokens > 0 else 0
    
    def unknown_rate(self, texts: list[str]) -> float:
        """Fraction of tokens that are UNK."""
        if self.unk_token is None:
            return 0.0
        total = 0
        unks = 0
        for text in texts:
            tokens = self.encode_fn(text)
            total += len(tokens)
            unks += sum(1 for t in tokens if t == self.unk_token)
        return unks / total if total > 0 else 0
    
    def token_length_stats(self, texts: list[str]) -> dict:
        """Distribution of token lengths (in characters)."""
        lengths = [len(t) for text in texts for t in self.encode_fn(text)]
        return {
            "mean": sum(lengths) / len(lengths) if lengths else 0,
            "min": min(lengths) if lengths else 0,
            "max": max(lengths) if lengths else 0,
        }
    
    def evaluate(self, texts: list[str]) -> dict:
        """Run all metrics."""
        return {
            "name": self.name,
            "fertility": self.fertility(texts),
            "compression_ratio": self.compression_ratio(texts),
            "unknown_rate": self.unknown_rate(texts),
            "token_lengths": self.token_length_stats(texts),
        }

print("TokenizerEvaluator class defined — reusable for any tokenizer.")

In [ ]:
# Evaluate all our tokenizers on a mixed benchmark
evaluators = [
    TokenizerEvaluator("GPT-2", lambda t: gpt2_tokenizer.tokenize(t)),
    TokenizerEvaluator("Custom BPE (8k)", lambda t: bpe_tokenizer.encode(t).tokens),
    TokenizerEvaluator("WordPiece (8k)", lambda t: wp_tokenizer.encode(t).tokens, unk_token="[UNK]"),
    TokenizerEvaluator("Unigram (8k)", lambda t: uni_tokenizer.encode(t).tokens, unk_token="[UNK]"),
    TokenizerEvaluator("Code BPE (8k)", lambda t: code_tokenizer.encode(t).tokens),
]

eval_texts = [
    "The transformer architecture revolutionized natural language processing.",
    "Attention is all you need — this paper introduced the paradigm shift.",
    "def forward(self, x): return self.linear(self.dropout(self.activation(x)))",
    "The patient presents with acute myocardial infarction and elevated troponin levels.",
    "According to Section 42(b) of the Internal Revenue Code, distributions from qualified plans.",
    "Neural networks learn hierarchical representations through backpropagation of gradients.",
    "The quick brown fox jumps over the lazy dog near the riverbank.",
    "Les réseaux de neurones apprennent des représentations hiérarchiques.",
]

eval_results = []
for evaluator in evaluators:
    result = evaluator.evaluate(eval_texts)
    eval_results.append({
        "Tokenizer": result["name"],
        "Fertility": f"{result['fertility']:.3f}",
        "Compression": f"{result['compression_ratio']:.2f}",
        "UNK Rate": f"{result['unknown_rate']:.4f}",
        "Avg Token Len": f"{result['token_lengths']['mean']:.1f}",
    })

print("Tokenizer Evaluation Results")
print("=" * 70)
print(pd.DataFrame(eval_results).to_string(index=False))

In [ ]:
# Visualize evaluation metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names = [r["Tokenizer"] for r in eval_results]
fertilities = [float(r["Fertility"]) for r in eval_results]
compressions = [float(r["Compression"]) for r in eval_results]
avg_lens = [float(r["Avg Token Len"]) for r in eval_results]

for ax, values, label, title, color in [
    (axes[0], fertilities, "Fertility (tokens/word)", "Fertility (lower = better)", "#3498db"),
    (axes[1], compressions, "Chars per token", "Compression Ratio (higher = better)", "#2ecc71"),
    (axes[2], avg_lens, "Avg token length (chars)", "Average Token Length", "#e74c3c"),
]:
    bars = ax.barh(names, values, color=color, edgecolor="white")
    ax.set_xlabel(label)
    ax.set_title(title)
    for i, v in enumerate(values):
        ax.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Cross-domain evaluation — how well does each tokenizer generalize?
domains = {
    "English prose": [
        "The history of artificial intelligence began in antiquity with myths and stories.",
        "Philosophers contemplated the possibility of artificial beings for centuries.",
        "The field was formally founded at a workshop at Dartmouth College in 1956.",
    ],
    "Python code": [
        "def train_model(self, epochs=10, lr=0.001): return self.fit(epochs, lr)",
        "model = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, 10))",
        "optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)",
    ],
    "Medical text": [
        "The patient presents with acute myocardial infarction and elevated troponin.",
        "Echocardiography revealed reduced left ventricular ejection fraction of 35%.",
        "Administer intravenous heparin and schedule percutaneous coronary intervention.",
    ],
    "French text": [
        "L'intelligence artificielle transforme notre compréhension du langage naturel.",
        "Les modèles de langue apprennent des représentations distribuées des mots.",
        "La tokenisation est une étape fondamentale du traitement automatique des langues.",
    ],
}

cross_domain_results = []
for evaluator in evaluators:
    row = {"Tokenizer": evaluator.name}
    for domain, texts in domains.items():
        row[domain] = f"{evaluator.fertility(texts):.2f}"
    cross_domain_results.append(row)

print("Cross-Domain Fertility (tokens per word — lower is better)")
print("=" * 80)
print(pd.DataFrame(cross_domain_results).to_string(index=False))

In [ ]:
# Vocabulary overlap analysis — how similar are our tokenizers?

def vocab_overlap(vocab_a: set, vocab_b: set) -> dict:
    """Compute vocabulary overlap statistics."""
    intersection = vocab_a & vocab_b
    union = vocab_a | vocab_b
    return {
        "size_a": len(vocab_a),
        "size_b": len(vocab_b),
        "intersection": len(intersection),
        "jaccard": len(intersection) / len(union) if union else 0,
        "overlap_a": len(intersection) / len(vocab_a) if vocab_a else 0,
        "overlap_b": len(intersection) / len(vocab_b) if vocab_b else 0,
    }

bpe_vocab = set(bpe_tokenizer.get_vocab().keys())
wp_vocab = set(wp_tokenizer.get_vocab().keys())
code_vocab = set(code_tokenizer.get_vocab().keys())
multi_vocab = set(multi_tokenizer.get_vocab().keys())

comparisons = [
    ("BPE vs WordPiece", bpe_vocab, wp_vocab),
    ("BPE vs Code BPE", bpe_vocab, code_vocab),
    ("BPE vs Multilingual", bpe_vocab, multi_vocab),
    ("Code vs Multilingual", code_vocab, multi_vocab),
]

print("Vocabulary Overlap Analysis")
print("=" * 65)
for name, va, vb in comparisons:
    stats = vocab_overlap(va, vb)
    print(f"\n{name}:")
    print(f"  Sizes: {stats['size_a']:,} vs {stats['size_b']:,}")
    print(f"  Shared tokens: {stats['intersection']:,}  |  Jaccard: {stats['jaccard']:.3f}")
    print(f"  A in B: {stats['overlap_a']:.1%}  |  B in A: {stats['overlap_b']:.1%}")

---

## 7. Saving, Loading, and Converting Tokenizers

After training a tokenizer, you need to serialize it for deployment. The `tokenizers` library saves to a single JSON file containing the full configuration: normalizer, pre-tokenizer, model (with all merge rules), post-processor, and decoder. You can also convert a raw `tokenizers.Tokenizer` into a Hugging Face `PreTrainedTokenizerFast` for seamless integration with the `transformers` training and inference pipeline.

In [ ]:
# Save, reload, and verify roundtrip
with tempfile.TemporaryDirectory() as tmpdir:
    save_path = os.path.join(tmpdir, "custom_bpe_tokenizer.json")
    bpe_tokenizer.save(save_path)
    print(f"Saved tokenizer ({os.path.getsize(save_path):,} bytes)")
    
    loaded_tokenizer = Tokenizer.from_file(save_path)
    
    # Verify roundtrip
    original_enc = bpe_tokenizer.encode(test_text)
    loaded_enc = loaded_tokenizer.encode(test_text)
    assert original_enc.ids == loaded_enc.ids, "Roundtrip failed!"
    print(f"Roundtrip verification: PASSED")

# Convert to HuggingFace PreTrainedTokenizerFast
with tempfile.TemporaryDirectory() as tmpdir:
    save_path = os.path.join(tmpdir, "tokenizer.json")
    bpe_tokenizer.save(save_path)
    
    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=save_path,
        bos_token="[CLS]", eos_token="[SEP]",
        unk_token="[UNK]", pad_token="[PAD]", mask_token="[MASK]",
    )
    
    encoded = hf_tokenizer(test_text, return_tensors=None)
    print(f"\nHF tokenizer output:")
    print(f"  input_ids:      {encoded['input_ids'][:10]}...")
    print(f"  attention_mask: {encoded['attention_mask'][:10]}...")
    print(f"  Decoded: {hf_tokenizer.decode(encoded['input_ids'])}")

---

## Summary

In this advanced notebook on tokenization, we covered the full practical toolkit for building, evaluating, and deploying custom tokenizers:

**Training custom tokenizers**: We used the Hugging Face `tokenizers` library to train BPE, WordPiece, and Unigram tokenizers from scratch. The four-stage pipeline (normalizer, pre-tokenizer, model, post-processor) gives you fine-grained control over every aspect of tokenization. Each algorithm has different tradeoffs: BPE is simple and effective, WordPiece uses likelihood-based merges, and Unigram provides probabilistic tokenization.

**Domain-specific tokenization**: General-purpose tokenizers waste tokens on domain text. We demonstrated this by comparing GPT-2's tokenizer against a code-specific tokenizer on Python code, measuring the compression ratio improvement. The fertility metric (tokens per word) provides a quantitative measure of this waste.

**Multilingual challenges**: The tokenization tax is real and measurable. Languages with non-Latin scripts or different morphological patterns can require 2-5x more tokens than English for the same semantic content. Balanced corpus sampling during training helps, but fundamental architectural choices (byte-level vs character-level) matter more.

**Byte-level BPE**: By operating on raw bytes (256 possible values) instead of Unicode characters, byte-level BPE guarantees zero unknown tokens for any input. We implemented it from scratch and showed how GPT-2's byte-to-unicode mapping works under the hood.

**Chat templates and special tokens**: Modern chat models rely on specific token patterns to structure multi-turn conversations. We examined the templates used by ChatML, Llama 3, and Mistral, and built a custom tokenizer with post-processing support.

**Evaluation metrics**: Fertility, compression ratio, and unknown token rate give us quantitative tools to compare tokenizers. Cross-domain evaluation reveals how well a tokenizer generalizes, and vocabulary overlap analysis helps decide whether to train a new tokenizer or reuse an existing one.

## Next Steps

Continue to **[expert.ipynb](./expert.ipynb)** where we will cover:

- Tokenizer-aware training: how the tokenizer interacts with model pretraining objectives
- Vocabulary pruning and extension for transfer learning scenarios
- SentencePiece internals and protobuf model format
- Tokenization for multimodal models (text + images + audio)
- Research frontiers: tokenizer-free models and learned segmentation